<a href="https://colab.research.google.com/github/Nicoledon/applied_machine_learning/blob/main/hw3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Programming 3 - Modeling Energy Usage

# Table of Contents

- [**Introduction**](#intro)
- [**Q1** - Weighted Linear Regression](#q1)
- [**Q2** - Iterative Training Methods](#q2)
    - [**Q2a** - Helper Functions for Mini-Batch Stochastic Gradient Descent](#q2a)
    - [**Q2b** - Implementing Mini-Batch Stochastic Gradient Descent for Weighted Linear Regression](#q2b)
    - [**Q2c** - Programming Written Plotting](#q2c) (Writeup)
    - [**Q2d** - Gradient Descent Plot Analysis](#q2d) (Writeup)
    - [**Q2e & Q2f** - Big-O Complexity Questions](#q2e) (Writeup)
- [**Q3** - Nonlinear Modeling](#q3)
    - [**Q3a** - Polynomial Features](#q3a)
    - [**Q3b** - K-Fold Cross Validation](#q3b)
    - [**Q3c** - Programming Written Plot](#q3c) (Writeup)

# Setup <a class="anchor" name="setup"></a>

**Note:** running the second cell in this section with the `curl` and `unzip` command might require you to replace some files if you have ran it before, so if that is the case, then click where the cursor is blinking and enter the command to replace all.

You'll need to run these cells, but you don't have to worry about their contents. You can look through them if you'd like of course.

In [ ]:
# Install otter-grader if needed

import importlib

if importlib.util.find_spec("otter") is None:
    !pip install otter-grader

if importlib.util.find_spec("sklearn") is None:
    !pip install scikit-learn

In [ ]:
# Copy additional files if needed
# Note: csv files can be loaded from the internet while np.load doesn't support reading from the internet

import os

if not os.path.isdir("tests") and "data.npz" not in os.listdir():
    !curl https://www.cs.cmu.edu/~10315/assignments/hw3_additional_files.zip --output hw3_additional_files.zip
    !unzip hw3_additional_files.zip

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsRegressor
import re

import otter

# Bump up the default font size for matplotlib
plt.rcParams.update({'font.size': 18})

In [ ]:
# Revert to older numpy repr strings to make otter-grader happy
if repr(np.float64()).startswith('np.float'):
     np.set_printoptions(legacy='1.25')

In [ ]:
grader = otter.Notebook()

# Tests

The tests that the Otter Grader uses are located in `test_case_code.py`. You can access this file by clicking the file icon on the left and double clicking on the python file (if you are using Google Colab). This file will contain the code for the test cases that the Otter Grader uses. You can copy and paste the code into code blocks and run it to debug your code. For the expected outputs, look at the corresponding Otter Grader tests.

# Introduction <a class="anchor" name="intro"></a>

You are a recently hired machine learning engineer at Pat's Eclectic Energy Enterprise ⚡. Currently, the company is having trouble determining how much power to produce for the day. If the enterprise produces too much energy, then it goes to waste and serves as extra company cost, and if they do not produce enough energy, then the company receives bad reviews and customers are unhappy.

As a result, Pat wanted to determine what is the best way to predict for energy consumption. After consulting his engineers, he realized that the temperature of the day could be a relatively strong indicator for energy consumption. Thus, since May till August, the company has been collecting data on the temperature of the day using a thermometer and the consumed energy for that day using expensive power meters.

As the machine learning engineer for the company, Pat would like you to develop a model for the consumption of energy with respect to the temperature feature that they collected data on. A visualization of the data can be seen below.

In [ ]:
def plotTempDataset(t):
    '''plotting the weighted linear regression model t=("summer"|"winter"|"full")'''
    data = np.load("data.npz")
    X, y, R = data[f"{t}_X"], data[f"{t}_y"], data[f"{t}_R"]
    y /= np.max(y)

    plt.figure(figsize=(15,10))
    plt.scatter(X, y, marker="P")
    plt.title(f"Temperature vs. Usage Data points")
    plt.xlabel("Temperature")
    plt.ylabel("Usage")
    plt.show()

plotTempDataset("summer")

Due to the linear relationship in the image, you believe that fitting a linear regression model would fare well. **Unfortunately**, Pat mentions that there is a problem with the dataset. When the day was extremely hot (over 80 degrees Fahrenheit), the power meters that measured the demand for data started malfunctioning. As a result, they produced measurements that are far more noisy than what is typically expected from the sensor.

That is an issue for linear regression. Linear regression assumes that your errors/sample noise is uniform across all datapoints, but instead, we have different variances for data points associated with higher temperatures. As a result, from your studies and superb understanding of machine learning theory, you decide that performing weighted linear regression would be a better fit. That way, you can model the change in variance of the dataset that you are trying to predict on. The dataset comes with sample weights where each sample has an associated non-negative real-value number representing how well the data model's the true relationship between temperature and usage. (a higher variance gives the data point a lower weight). Below is a colored visualization of the data showing the data points where the sensor had low variance and the data points where the sensor had high variance.

In [ ]:
def plotTempDatasetWeighted(t):
    '''plotting the weighted linear regression model t=("summer"|"winter"|"full")'''
    data = np.load("data.npz")
    X, y, R = data[f"{t}_X"], data[f"{t}_y"], data[f"{t}_R"]

    idxs = X >= 80
    X_low = X[~idxs]
    X_high = X[idxs]

    y_low = y[~idxs]
    y_high = y[idxs]

    plt.figure(figsize=(15,10))
    plt.scatter(X_low, y_low, marker="P", c="blue", label="Low Variance (Weight=1)")
    plt.scatter(X_high, y_high, marker="P", c="orange", label="High Variance (Weight=0.25)")
    plt.title(f"Temperature vs. Usage Data points")
    plt.xlabel("Temperature")
    plt.ylabel("Usage")
    plt.legend()
    plt.show()

plotTempDatasetWeighted("summer")

As a result, you decide that you'll attempt to model the energy consumption as a function of a temperature feature using a weighted linear regression model in multiple ways. You'll gain a better understanding of how linear regression models are trained, and by trying different optimization techniques, you'll also develop intuition for various optimization algorithms such as the variants of gradient descent and understand the tradeoffs that come with them.

Unfortunately, you don't have access to any libraries for training weighted linear regression models, and as a result, you'll have to implement the model on your own. Below, you've written an abstract class `Model` for defining the operations that are used with a machine learning model, such as `__init__` (or initializing) it with a specific hyperparameter configuration, `fit`-ing (or training) it given a dataset, `predict`-ing it on a new dataset, or `score`-ing it on a new set of samples with corresponding features. (Note: this is a similar interface that the `sklearn` Python machine learning package implements their machine learning models in)

Using this abstract class, you'll implement weighted linear regression in multiple ways that will help you gain a better understanding of linear regression models as well as various optimization techniques.

**Terminology Alert:** fitting a model to a dataset and training a model on a dataset are the same thing!

**Note:** You don't have to implement anything in the `Model` class!

In [ ]:
# NOTE: this is an abstract class that is only meant for you to get an idea for a machine learning model does
#       you do not need to implement/modify any functions here!

class Model:
    def __init__(self, *args):
        """initialize the machine learning model -- pass arguments that are hyperparameters for training"""
        pass

    def fit(X, y, R=None):
        """ train the model on the dataset of samples X and their corresponding labels y
            if R given, then samples have weights
            Note: fitting is the same thing as training!
        """
        pass

    def predict(X):
        """predict on a new dataset of samples X"""
        pass

    def score(X, y):
        """ evaluate how well the model performs on a dataset of samples X with labels y
            compare the predicted values of the model with y and evaluate some loss or accuracy metric
        """
        pass

---
# Question 1 - Weighted Linear Regression <a class="anchor" name="q1"></a>

Recall that you are using weighted linear regression as your model of choice. The reason for this is that we would like to weight the values of the the more-noisy datapoints with less importance (smaller $r_i$). Conversely, we would like to weight the values of less-noisy datapoints with more importance (larger $r_i$) because they have a better indication of what we would like to model.

As a result, given a dataset of points (i.e. temperature values) in the form of the design matrix, $X$, which is (N,2) (because one a column of bias values and a column of temperatures), their corresponding targets (i.e. usage values) $\mathbf{y}$ which is (N,1), and weights for each of the samples $R$ which is (N,N) ($R$ is a __diagonal__ matrix that contains weights for the *i*-th sample at $r_{ii}$), we can write down the weighted least squares loss of the weighted linear regression model as follows.

$$J(\mathbf{w})  = \frac{1}{N}(\mathbf{y}-X\mathbf{w})^{\top}R(\mathbf{y}-X\mathbf{w})$$

where $\mathbf{w}$ denotes the parameters for the linear regression model. You will be using your derivation of the closed-form solution and the gradient of this loss with respect to the weights to implement weighted least squares linear regression in the following iterative and non-iterative methods.

1. Closed-Form Solution
2. Gradient Descent
3. Stochastic Gradient Descent
4. Mini-Batch Stochastic Gradient Descent

**Note:** for $R$, we are going to give you an `(N,)` element `np.ndarray` which you can then convert to an (N,N) matrix. We recommend looking into `np.diag` as a useful function to do so.

**Additional Note:** We will represent the raw dataset of features as `Xraw` while we will represent the design matrix created from the raw features as `X`. Furthermore, for all matrix multiplications throughout this assignment, **do not use** `np.matmul`. Instead, **use** `@`.

First, we'll get started with training weighted linear regression through a closed-form solution using the theory for the problem that you solved in the written portion of this assignment. First, in order to help with training linear regression models, implement the helper function below.

In [ ]:
def makeDesignMatrix(Xraw):
    """ Creates the design matrix from matrix Xraw by concatenating a
        column of 1's to the left of the array
        Used by linear regression models to create a bias feature

        >>> a
        array([[ 4.,  7.],
               [ 3.,  7.],
               [12.,  7.]])
        >>> makeDesignMatrix(a)
        array([[ 1.,  4.,  7.],
               [ 1.,  3.,  7.],
               [ 1., 12.,  7.]])

        Input:
        Xraw: np.ndarray of shape (N,M)

        Returns:
        X: design matrix made from Xraw
    """
    one = np.ones([Xraw.shape[0],1])
    return np.hstack([one,Xraw])


In [ ]:
# make sure the outputs are what you expect!
Xraw = np.array([[1,2],[3,4],[5,6]])
X = makeDesignMatrix(Xraw)
X

In [ ]:
Xraw = np.array([[0,0,0,2],[2,3,5,9],[0,3,0,8],[0,3,0,7]])
X = makeDesignMatrix(Xraw)
X

Now, with having implemented the matrix to create our design matrix which adds a 1's feature for creating a bias in our linear regression model, we will now train our weighted linear regression model with the closed form solution.

### Implement `.fit` and `.predict` of Class `WeightedLinReg_ClosedForm`

**Note:** if you are running your code and the test cases are not passing even if you believe that you are passing the test cases, try resetting your notebook and clicking `Run All`.

In [ ]:
class WeightedLinReg_ClosedForm(Model):
    # closed form solution to weighted linear regression
    def __init__(self):
        # Do not need to implement anything here
        pass

    def fit(self, Xraw, y, R):
        """ Calculates the closed-form solution parameters for weighted linear regression
            Note: be aware of the shapes of your input when you are doing NumPy operations!

            Input:
            Xraw: np.ndarray of shape (N, M) representing N data points each consisting of M features
            y: np.ndarray of shape (N,1) representing the targets for the each of the N data points
                - data point X[i,:] has feature y[i]
                - y is a column vector
            R: np.ndarray of shape (N,) representing the weights for each of the samples

            No output -- but after this function call, self.params must store the most up-to-date model parameters
        """
        R = np.diag(R)
        Xraw = makeDesignMatrix(Xraw)
        self.params = np.linalg.inv(Xraw.T @ R @ Xraw) @ Xraw.T @ R @ y

    def predict(self, Xraw):
        """ Calculates the predicted target values of the datapoints in Xraw and returns as a column vector

            Note: make sure to convert Xraw to the design matrix before making your predictions
            Note: predict can only be used after .fit is called, use self.params to perform prediction
            Note: weights are not necessary at prediction time -- only at training

            Input:
            Xraw: np.ndarray of shape (N, M) representing N data points each consisting of M features

            Output:
            y_hat: np.ndarray of shape (N,1) representing the predicted target values for the N data points in X
                - y_hat[i] is the predicted target value for data point X[i,:]

        """
        y_hat = makeDesignMatrix(Xraw) @ self.params
        return y_hat

Feel free to write your own code to test and visualize your `WeightedLinReg_ClosedForm` model implementation in as many cells as you would like below!

In [ ]:
Xraw = np.array([[1,1],[2,2],[3,3.5]])
y = 3 + Xraw @ np.array([1,1]).reshape((-1,1))
R = np.array([1,1,1])

lr = WeightedLinReg_ClosedForm()
lr.fit(Xraw, y, R)
np.round(lr.params, 4)

In [ ]:
Xraw = np.array([[1,1],[2,2],[3,3.5]])
y = 3 + Xraw @ np.array([1,1]).reshape((-1,1))
R = np.array([1,1,1])

lr = WeightedLinReg_ClosedForm()
lr.fit(Xraw, y, R)
predictions = lr.predict(np.array([[1,2],[3,3],[4,5]]))
np.round(predictions, 4)

In [ ]:
grader.check("Q1")

## Visualizing Weighted Linear Regression
Run the `plotWLR` function to visualize your model on the dataset that you were provided by the company and ensure that the weighted linear regression model appropriately fit to the data.

In [ ]:
def plotWLR():
    '''plotting the weighted linear regression model'''
    data = np.load("data.npz")
    Xraw, y, R = data["summer_X"], data["summer_y"], data["summer_R"]
    R = np.diag(R)

    lr = WeightedLinReg_ClosedForm()
    lr.fit(Xraw, y, R)

    plt.figure(figsize=(15,10))
    plt.scatter(Xraw, y, marker="P")
    XSample = np.linspace(np.min(Xraw), np.max(Xraw), 400).reshape((-1, 1))
    plt.plot(XSample, lr.predict(XSample), linestyle="solid", linewidth=4, color="lightgreen")
    np.set_printoptions(precision=4)
    plt.title(f"Model Type: Closed Form WeightedLinReg b={lr.params[0]} w={lr.params[1]}")
    plt.xlabel("Temperature")
    plt.ylabel("Usage")
    plt.show()

In [ ]:
plotWLR()

### Submit your code to Gradescope early and often

There is no limit on the number of submissions to Gradescope, so as you complete parts of the assignment it is a really good idea to save your notebook and upload it to Gradescope.

Not all of the tests are included in the local autograder. Some of the tests are "hidden" and only run in the server autograder on Gradescope.

Before continuing with the rest of the assignment, go ahead and save your notebook (or click File->Download->Download .ipynb) and then upload your hw3.ipynb file to Gradescope under assignment HW3 (programming).

---
# Question 2 - Iterative Training Methods <a class="anchor" name="q2"></a>
- [**Q2a** - Helper Functions for Mini-Batch Stochastic Gradient Descent](#q2a)
- [**Q2b** - Implementing Mini-Batch Stochastic Gradient Descent for Weighted Linear Regression](#q2b)
- [**Q2c** - Programming Written Plotting](#q2c) (Writeup)
- [**Q2d** - Gradient Descent Plot Analysis](#q2d) (Writeup)
- [**Q2e & Q2f** - Big-O Complexity Questions](#q2e) (Writeup)

Now that you've implemented the weighted linear regression model, you are interested in finding different training methods for weighted linear regression, specifically through different variants of gradient descent.

There are 3 main different types of gradient descent methods that you will be implementing:

1. Gradient Descent (GD) - the entire dataset is used to estimate the gradient before each update
2. Stochastic Gradient Descent (SGD) - a single sample is used to estimate the gradient before each update
3. Mini-Batch Stochastic Gradient Descent (MBSGD) - a batch of samples is used to estimate the gradient before each update based on a batch size

**However,** instead of implementing the learning algorithms for all three gradient descent methods, notice that there is a relationship between MBSGD and the other two gradient descent methods, GD and SGD, with respect to the batch size. So, if you can train a model with MBSGD, you can already train it with GD and SGD!

With this understanding, you decide to implement the mini-batch stochastic gradient descent algorithm, but first, there are some helper functions necessary to do so.

## Question 2a - Helper Functions for Mini-Batch Stochastic Gradient Descent <a class="anchor" name="q2a"></a>

`shuffleDataset` will be given to you. When shuffling a dataset use this function, do not use any other random function as that will cause testing difficulties. Note: `shuffleDataset` doesn't actually appropriately shuffle the dataset for you by creating a random permutation of data. It simply cycles the array by one element to simulate shuffling. In practice, we would actually run the code that is in the comments below.

In [ ]:
def shuffleDataset(X, y, R):
        """ Shuffles the dataset into a random permutation

            Input:
            X: np.ndarray of shape (N,M) that is the design matrix used for training the model
            y: np.ndarray of shape (N,1) that contains the target values for each sample in the design matrix
            R: np.ndarray of shape (N,) that is an array containing the weights of each sample

            Output:
            XShuffled: np.ndarray of shape (N,M) shuffled version of X
            yShuffled: np.ndarray of shape (N,1) shuffled version of y
            RShuffled: np.ndarray of shape (N,) shuffled version of R
        """
        idxs = np.arange(X.shape[0])
        np.random.seed(10315) # NOTE: required for test cases, but in practice this line should be removed
        np.random.shuffle(idxs)
        XShuffled, yShuffled, RShuffled = X[idxs, :], y[idxs, :], R[idxs]

        return X[idxs, :], y[idxs, :], R[idxs]

### Implement `initParams`

Any gradient descent method needs a starting point at which to compute the gradient. Define the function `initParams` which initializes the parameters that will be learned via gradient descent.

In [ ]:
def initParams(numParameters):
    """ Initializes the parameters for a model to be trained with gradient descent.
        Iniitialize all parameter values to 1

        When calling this function, make sure to consider the bias

        Input:
        numParameters: the number of parameters that the linear model will have

        Output:
        params: an (numParameters,1) column vector of ones
    """
    params = np.ones([numParameters, 1])
    return params

### Implement `computeBatches`
With mini-batch stochastic gradient descent, at the start of each epoch, the dataset is shuffled and then partitioned into groups of batches. Implement this functionality in the function below.

In [ ]:
def computeBatches(X, y, R, batchSize):
    """ Places the dataset provided into a list of batches of the dataset according to batch size

        Input:
        X: np.ndarray of shape (N,M) that is the design matrix used for training the model
        y: np.ndarray of shape (N,1) that contains the target values for each sample in the design matrix
        R: np.ndarray of shape (N,) that is an array containing the weights of each sample
        batchSize: (int) the size of the batches

        NOTE: if the size of the dataset is not divisible by the batchSize, leave the last batch
        to be the size of the remainder of the dataset (i.e X.shape[0] % batchSize)

        Output:
        batches: list of 3-tuples of np.ndarrays Xsub, ysub, and Rsub
            - batches[i] = (Xsub, ysub, Rsub) for samples from batchSize * i to batchSize * (i + 1)
            - the size of each batch is either batchSize or X.shape[0] % batchSize (for the edge case)
    """
    batches = []
    N = X.shape[0] // batchSize
    for i in range(N):
        tup = (X[i * batchSize : (i + 1) * batchSize] , y[i * batchSize : (i + 1) * batchSize ]  , R[i * batchSize : (i + 1) * batchSize])
        batches.append(tup)
    num = X.shape[0] % batchSize
    if num != 0:
        tup = (X[-num:] , y[-num:] , R[-num:])
        batches.append(tup)
    return batches

### Implement `computeGradient`
Finally, regardless of the method of gradient descent, the gradient of the loss of the weighted dataset w.r.t. the parameters is always the same. Using your theory, implement a method to compute the gradient of the weighted loss function with respect to the parameters for weighted linear regression.

In [ ]:
def computeGradient(p, X, y, R):
    """ Computes the gradient of the loss w.r.t. the parameters p for a linear model.
        The loss is the weighted mean squared error function


        Inputs:
        p: np.ndarray of shape (M,1) that contains the parameters for some linear regression model
        X: np.ndarray of shape (N,M) that is the design matrix used for training the model
        y: np.ndarray of shape (N,1) that contains the target values for each sample in the design matrix
        R: np.ndarray of shape (N,) that is an array containing the weights of each sample

        Outputs:
        g: np.ndarray of shape (M,1) that contains the gradients calculated from the data
            - g[i] corresponds to the gradient of the parameter p[i]
            - Note: the weighted mean squared error function is the average of the weighted squared error for each datapoint
    """
    N = y.shape[0]
    R = np.diag(R)
    return 2 / N  * (X.T @ R @ X @ p - X.T @ R @ y)




In [ ]:
X = np.array([[1,1,-1],[1,2,-2],[1,3,-3],[1,4,-4],[1,5,-5],[1,6,-6]])
y = np.array([1,2,3,4,5,6]).reshape((-1,1))
R = np.array([0.25,0.25,1,0.75,0,0])
batchSize = 2

batches = computeBatches(X, y, R, batchSize)

In [ ]:
params = initParams(3)
params

In [ ]:
X = np.array([[1,1,-1],[1,2,-2],[1,3,-3],[1,4,-4],[1,5,-5],[1,6,-6]])
y = np.array([1,2,3,4,5,6]).reshape((-1,1))
R = np.array([0.25,0.25,1,0.75,0,0])
batchSize = 2

batches = computeBatches(X, y, R, batchSize)
batches[1][1]

In [ ]:
p = np.array([-1,1]).reshape((-1,1))
X = np.array([[1,-1],[1,2]])
y = np.array([-3,3]).reshape((-1,1))
R = np.array([1,0.5])

grad = computeGradient(p, X, y, R)
grad

In [ ]:
grader.check("Q2a")

## Question 2b - Implementing Mini-Batch Stochastic Gradient Descent for Weighted Linear Regression <a class="anchor" name="q2b"></a>
Finally, with all of the helper methods in place, we can now use mini-batch stochastic gradient descent to find the optimal parameters for weighted linear regression. Now, you can implement the `WeightedLinReg_MBSGD` class.

For the purposes of grading and nice plotting, below is the definition and implementation of the `TrainingLogger` class that defines a way to store the parameters and training error over multiple iterations of gradient descent and view the training process. Overall, the details of this class are unimportant, but when implementing mini-batch stochastic gradient descent, **define an object of this class at the start of training, log the initialized parameters and error, and then, for every update to the parameters in `WeightedLinReg_MBSGD.fit(...)`, log their values after the update**.

Note: you will need to calculate the weighted mean-squared error to pass into the `error` parameter of `TrainingLogger.log(params, error)`. The weighted mean-square error is defined below

$$MSE = \frac{1}{N} \sum_{i=1}^N R_{i,i} \left(\mathbf{y}^{(i)} - \hat{\mathbf{y}}^{(i)}\right)^2$$

You will implement this loss function in the helper function `getWeightedMSE(...)`

Note: do not modify the `TrainingLogger` class!

In [ ]:
class TrainingLogger:
    def __init__(self, Xraw, y, R):
        """ Initializes the training logger with the specific weighted dataset (X, y, R)
            Every call to .log(params) will evaluate the given params against this dataset
            using the MSE metric

            Inputs:
            Xraw: np.ndarray of shape (N, M) representing N data points each consisting of M features
            y: np.ndarray of shape (N,1) representing the targets for the each of the N data points
            R: np.ndarray matrix of shape (N,) that is an array containing the weights of each sample
        """
        self.data = (Xraw, y, np.diag(R))
        self.allIterParams = []
        self.allIterErrors = []

    def log(self, params, error):
        """ Log the parameter values and calculate their error w.r.t. the params

            Inputs:
            params: np.ndarray of shape (M+1,1) that contains the parameters for some linear regression model
                - note that the +1 is due to the bias parameter
            error: (float) represents the weighted mean-squared error of the model on the training dataset
        """
        self.allIterParams.append(np.zeros(params.shape) + params)
        self.allIterErrors.append(error)

### Implement `getWeightedMSE(...)`

In [ ]:
def getWeightedMSE(Xraw, y, R, params):
    """ Calculates the weighted mean squared error of the given dataset on the given set of parameters

        Input:
        Xraw: np.ndarray of shape (N, M) representing N data points (i.e. samples) each consisting of M features
        y: np.ndarray of shape (N,1) representing the true target values for the N data points in Xraw
        R: np.ndarray of shape (N,) representing the weights for each of the N data points in Xraw
        params: np.ndarray of shape (M+1,1) that are the current parameters of the model to evaluate

        Output:
        error: float equal to the weighted mean-squared error of the dataset on the parameters
    """
    N = y.shape[0]
    R = np.diag(R)
    X = makeDesignMatrix(Xraw)
    y_hat = X @ params
    error = np.sum(R * (y - y_hat) ** 2) / N
    return error

### Implement `.fit(...)`, and `.predict(...)` of `WeightedLinReg_MBSGD`
Implement the following functions listed above that are missing their implementation.

In [ ]:
class WeightedLinReg_MBSGD(Model):

    def __init__(self, lr=0.02, epochs=500, batchSize=256):
        """ Initializes a model that can be trained
            with the mini-batch gradient descent algorithm using the .fit(...) method

            gradient descent algorithms can be:
            1. regular gradient descent (GD)
            2. stochastic gradient descent (SGD)
            3. mini-batch stochastic gradient descent (MBSGD)
            (note: MBSGD is a generalization of GD and SGD, how?)

            Input:
            lr: (float) the learning rate to use during gradient descent
            epochs: (int) the number of iterations to perform gradient descent
                - note: the number of epochs is the number of times you iterate
                        over the entire dataset
            batchSize: (int) the size of each batch in training
        """
        # Note: do not need to modify __init__ method
        self.lr = lr
        self.epochs=epochs
        self.batchSize = batchSize

    def fit(self, Xraw, y, R):
        """ Performs mini-batch stochastic gradient descent to train the model
            for self.epochs iterations and with a learning rate of self.lr and a batch size of self.batchSize

            1) This function initializes parameters, constructs the design matrix, and performs multiple epochs.
            Each epoch will shuffle the data, batch it, and then iterate through the batches, and update the model
            parameters using that batch using the gradient of the loss w.r.t. the parameters.

            2) Additionally, this function will log the value of the parameters at the start of training and
            once after every single update to the parameter values, and store the log in self.logger.

            Note: Make sure to shuffle and then batch the original dataset that you were given. Do not reshuffle a
            version of the dataset that has been already shuffled.

            Input:
            Xraw: np.ndarray of shape (N,M) that is the dataset of N data points each with M features
            y: np.ndarray of shape (N,1) that contains the target values for each sample in the design matrix
            R: np.ndarray of shape (N,) that is an array containing the weights of each sample

            No output -- self.params and self.logger must be defined after this function has completed
                - self.params must equal the fitted parameters after performing mini-batch stochastic gradient descent
                - self.logger must contain the training logger that has logged 1 + numBatches * self.epochs times
                    - once after the parameters were initialize
                    - numBatches * self.epochs times after the parameters were updated
        """
        self.params, self.logger = None, None
        self.params= initParams(Xraw.shape[1] + 1)
        self.logger = TrainingLogger(Xraw, y, R)
        for i in range(self.epochs):
           x_train , y_train , r_train = shuffleDataset(Xraw, y, R)
           batches = computeBatches(x_train, y_train, r_train, self.batchSize)
           for batch in batches:
               Xsub, ysub, Rsub = batch
               biasX = makeDesignMatrix(Xsub)
               g = computeGradient(self.params, biasX, ysub, Rsub)
               self.params -= self.lr * g
               error = getWeightedMSE(Xsub, ysub, Rsub, self.params)
               self.logger.log(self.params, error)

    def predict(self, Xraw):
        """ Calculates the predicted target values of the datapoints in Xraw. Should only be called after
            .fit is called, use self.params to perform prediction

            Additional Note: sample weights are not necessary at prediction time -- only at training

            Input:
            Xraw: np.ndarray of shape (N, M) representing N data points each consisting of M features

            Output:
            y_hat: np.ndarray of shape (N,1) representing the predicted target values for the N data points in Xraw
        """
        x = makeDesignMatrix(Xraw)
        return x @ self.params

    def getTrainingLog(self):
        """ Returns the training logger that contains the information accumulated during training
            .log has been called on it for 1 + self.epochs * number_parameter_updates
                - once when the parameters were first initialized
                - once after every single update to the parameters
            Only callable after training
        """
        return self.logger

### Implement `trainMultipleModels()`
Now that you have implemented the `WeightedLinReg_MGSD` class, we will define a function to train 3 weighted linear regression models on the same dataset but each using a different variant of gradient descent.
1. For each variant, use a learning rate of 0.02
2. For each variant, train for 500 epochs  
3. The first should use gradient descent, the second should use stochastic gradient descent, and the third should use mini-batch stochastic gradient descent with a batch size of 256

In [ ]:
def trainMultipleModels(Xraw, y, R):
    """ Trains multiple weighted linear regression models using various gradient descent methods
        Given the training dataset as input, trains 3 models of type WeightedLinReg_MBSGD

        All models are trained with 500 epochs and a learning rate of 0.02.
        However, the first model is trained via gradient descent. The second model is trained via
        stochastic gradient descent. The third model is trained via mini-batch stochastic gradient descent
        with a batch size of 256.

        Input:
        Xraw: np.ndarray of shape (N,M) that is the dataset of N data points each with M features
        y: np.ndarray of shape (N,1) that contains the target values for each sample in the design matrix
        R: np.ndarray of shape (N,) that is an array containing the weights of each sample

        Output:
        gd: WeightedLinReg_MBSGD trained using gradient descent
        sgd: WeightedLinReg_MBSGD trained using stochastic gradient descent
        mbsgd: WeightedLinReg_MBSGD trained using mini-batch stochastic gradient descent with batch size 256
    """

    gd= WeightedLinReg_MBSGD(lr=0.02, epochs=500, batchSize=Xraw.shape[0])
    gd.fit(Xraw , y , R)
    sgd= WeightedLinReg_MBSGD(lr=0.02, epochs=500, batchSize=1)
    sgd.fit(Xraw,y,R)
    mbsgd = WeightedLinReg_MBSGD(lr=0.02, epochs=500 ,batchSize=256)
    mbsgd.fit(Xraw,y,R)
    return gd, sgd, mbsgd


In [ ]:
Xraw = np.array([[-1,1],[1,-2,],[3,9],[-4,5],[1,1]])
params = np.array([3,-1,1]).reshape(-1,1)
y = params[2,:] + Xraw @ params[:2,:]
R = np.array([0.5,1,1,1,0.25])

err = getWeightedMSE(Xraw, y, R, np.array([-3,1,1]).reshape(-1,1))
err

In [ ]:
Xraw = np.array([[1,1],[2,2],[3,3.5]])
y = 3 + Xraw @ np.array([1,1]).reshape((-1,1))
R = np.array([1,1,1])

lr = WeightedLinReg_MBSGD(lr=0.5, epochs=1, batchSize=1)
lr.fit(Xraw, y, R)
predictions = lr.predict(np.array([[1,2],[3,3],[4,5]]))
np.round(predictions, 4)

In [ ]:
Xraw = np.array([[1,1],[2,2],[3,3],[2,2.5]])
y = np.array([-1,-2,-3,-2.3]).reshape((-1,1))
R = np.array([1,1,1,0.5])

lr = WeightedLinReg_MBSGD(lr=0.001, epochs=3, batchSize=2)
lr.fit(Xraw, y, R)
np.round(lr.params, 4)

In [ ]:
grader.check("Q2b")

These are just some helper functions to plot the data and the models that you've implemented. Feel free to look into the data, but make sure that you do not modify these functions as they will be used for the written portion of the programming assignment.

In [ ]:
def loadGDData():
    data = np.load("data.npz")
    Xraw, y, R = data["summer_X"], data["summer_y"], data["summer_R"]
    Xraw /= np.max(Xraw) # we scale the dataset to prevent the gradients from exploding
    y /= np.max(y)
    R = np.diag(R)

    return Xraw, y, R

def plotGDModels(X, y, R, gd, sgd, mbsgd):
    '''plotting the GD, SGD, MB-SGD models'''

    # plot the fitted functions
    models = [gd, sgd, mbsgd]
    fig, axes = plt.subplots(3, 1, figsize=(10,15), sharex=True, constrained_layout=True)
    X_sample = np.array([np.min(X), np.max(X)]).reshape((-1, 1))
    colors = ["orange", "green", "purple"]
    labels = ["gd", "sgd", "mbsgd"]

    for i in range(len(models)):
        axes[i].scatter(X, y, marker="P")
        axes[i].plot(X_sample, models[i].predict(X_sample), linestyle="solid", linewidth=4, color=colors[i])
        axes[i].set_title(f"Model Type:{labels[i]}, b={models[i].params[0]}, w={models[i].params[1]}")
        axes[i].set_xlabel("Temperature: scaled to [0,1]")
        axes[i].set_ylabel("Usage: scaled to [0,1]")

    plt.show()

def plotGDLosses(X, y, R, gd, sgd, mbsgd):
    '''plot the error rates between the 3 iterative models'''

    # plot the error rates of the models
    models = [gd, sgd, mbsgd]
    colors = ["orange", "green", "purple"]
    labels = ["gd", "sgd", "mbsgd"]

    epochs = 500
    epochsToShow = 4

    plt.figure(figsize=(10,10))

    for m, c, l in zip(models, colors, labels):
        logger = m.getTrainingLog()
        log = logger.allIterErrors
        samplesPerEpoch = (len(log)-1) // epochs
        nPoints = epochsToShow * samplesPerEpoch
        x = np.linspace(0, epochsToShow, nPoints+1)
        plt.plot(x, log[:nPoints+1], color=c, label=l)

    plt.xlabel("Epoch #")
    plt.ylabel("Loss (Weighted MSE)")
    plt.title("Loss Over Epochs for Iterative Training Methods")
    plt.legend()
    plt.show()

## Q2c - Programming Written Question <a class="anchor" name="q2c"></a>

Run the two plots below, and paste them into the writeup of the programming section of the homework for question **Q2c**. Afterwards, answer the following analysis questions in **Q2d through Q2f**.

Note: you may have gotten different parameter values for the closed form solution of weighted linear regression in comparison to the various gradient descent methods. This is perfectly fine because you will notice that the parameters for gradient descent will converge to the closed form solution if they trained for more and more epochs (around 5000 epochs). If you would like to test this out, change the number of epochs that `WeightedLinReg_GD` and `WeightedLinReg_MB_SGD` are trained for to 5000

In [ ]:
# note: this function will take some time due to a larger number of epochs used for training
#       and the logging that is happening on every parameter update (it's especially time-consuming for SGD)
Xraw, y, R = loadGDData()
gd, sgd, mbsgd = trainMultipleModels(Xraw, y, R)

In [ ]:
plotGDModels(Xraw, y, R, gd, sgd, mbsgd)

In [ ]:
plotGDLosses(X, y, R, gd, sgd, mbsgd)

## Q2d - Gradient Descent Analysis <a class="anchor" name="q2d"></a>
Analyze the gradient descent plots and answer question **Q2d** in the writeup of this assignment.

## Q2e and Q2f - Big-O Analysis <a class="anchor" name="q2e"></a>
Analyze the big-O complexity of closed form weighted linear regression and compare it to the complexity of a single iteration of gradient descent in **Q2e** and **Q2f** of the writeup of this assignment.

### Submit your code to Gradescope early and often

There is no limit on the number of submissions to Gradescope, so as you complete parts of the assignment it is a really good idea to save your notebook and upload it to Gradescope.

Not all of the tests are included in the local autograder. Some of the tests are "hidden" and only run in the server autograder on Gradescope.

Before continuing with the rest of the assignment, go ahead and save your notebook (or click File->Download->Download .ipynb) and then upload your hw3.ipynb file to Gradescope under assignment HW3 (programming).

# Question 3 - Nonlinear Modeling <a class="anchor" name="q3"></a>

- [**Q3a** - Polynomial Features](#q3a)
- [**Q3b** - K-Fold Cross Validation](#q3b)
- [**Q3c** - Programming Written Plot](#q3c) (Writeup)

With your success modeling the summer dataset, the company decided to collect more data throughout the year to get a better understanding of consumer demand depending on the temperature throughout the day. Earlier you were working with data collected from May to August. Now the company gathered data throughout the rest of the year, September to April. Now with data from the full year you can now create a model that is made from temperatures recorded throughout the whole year. Below, you can find the updated dataset, distinguished by the data collected from the summmer (May to August) and the winter (September to April).

In [ ]:
def plotDatasetSeason():
    '''plotting the weighted linear regression model t=("summer"|"winter"|"full")'''
    data = np.load("data.npz")

    X_summer, y_summer = data["summer_X"], data["summer_y"]
    X_winter, y_winter = data["winter_X"], data["winter_y"]

    plt.figure(figsize=(15,10))
    plt.scatter(X_summer, y_summer, marker="P", c="sandybrown", label="Summer")
    plt.scatter(X_winter, y_winter, marker="P", c="lightskyblue", label="Winter")
    plt.title(f"Temperature vs. Usage Data points")
    plt.xlabel("Temperature")
    plt.ylabel("Usage")
    plt.legend()

plotDatasetSeason()

**Oh no!** We were using linear models to model our dataset; however, after collecting data for the whole year, our dataset appears to have a non-linear relationship. Using a vanilla linear regression on its own may not model this dataset that well. As a result, you will take a first look at feature engineering, where you will engineer polynomial features from your dataset, and use that for linear regression. For example, typically in linear regression, you are modelling the following function (assume 1 feature and 1 target)

$$\mathbf{y}=\phi_0 + \phi_1\mathbf{x}$$

where $\phi_0$ and $\phi_1$ are your parameters. However, if you were to engineer features for up to the third degree polynomial, you would now be modelling this equation.

$$\mathbf{y} = \phi_0 + \phi_1\mathbf{x} + \phi_2\mathbf{x}^2 + \phi_3\mathbf{x}^3$$

Note, while the data that you are using represents polynomial features, the actual function is still linear w.r.t to the polynomial features. You are performing a linear combination of the polynomial features. As a result, by creating polynomial features, you can still use linear regression to model this dataset!

## Question 3a - Polynomial Features <a class="anchor" name="q3a"></a>

Before you can start implementing the weighted linear regression model with polynomial features, we first need to engineer polynomial features from our dataset.

In [ ]:
def makePolynomialDesignMatrix(Xraw, degree):
    """ Creates polynomial features from Xraw by concatenating a
        column of 1's to the left of the array and concatenating higher degree
        columns of the original columns to the right

        >>> a
        array([[ 4,  7],
               [ 3,  8],
               [12,  9]])
        >>> makePolynomialDesignMatrix(a, 3)
        array([[1,   4,    7,   16,   49,   64,  343],
               [1,   3,    8,    9,   64,   27,  512],
               [1,  12,    9,  144,   81, 1728,  729]])

        Input:
        Xraw: np.ndarray of shape (N,M)
        degree: int such that degree >= 1
        Returns:
        polynomial_matrix: matrix with polynomial features up to degree (N, 1+M*degree)
    """
    ...

In [ ]:
A = np.array([[4,7],[3,8],[12,9]])
X = makePolynomialDesignMatrix(A, 3).astype(np.int64)
X

In [ ]:
A = np.array([[1,1,1],[2,2,2],[3,3,3],[4,4,4]])
X = makePolynomialDesignMatrix(A, 2)
X

### Implement `.fit`, `.predict`, and `.score` for `PolyWeightedLinReg_ClosedForm`
Implement the closed form solution to the weighted linear regression classifier below; however, this time use polynomial features for the given input degree `self.degree` in the class. Furthermore, implement the function for prediction as well.

Finally, because you would like to compare this model with other models when doing model selection, we will have you compute the score function so that you can calculate how well the model is performing on an input dataset. However, instead of calculating the usual mean-squared error, we are going to have you calculate a new metric that is useful for evaluating regression models called the **coefficient of determination**, $R^2$, which is defined below.

$$R^2 = 1 - \frac{RSS}{TSS}$$

where the residual sum of squares is $RSS = \sum_{i=1}^N(y^{(i)} - f(x^{(i)}))^2$ and total sum of squares is $TSS =  \sum_{i=1}^N(y^{(i)} - \bar{y})^2$ where $\bar{y} = \frac{1}{N} \sum_{i=1}^Ny^{(i)}$. Implement this metric in the `.score` method for `PolyWeightedLinReg_ClosedForm`. The coefficient of determination is a metric that is maximized at value 1, and a higher value represents a model with a better fit to the dataset. We are using the coefficient of determination because it is a commonly used metric for evaluating model performance on various regression models and is commonly used throughout some machine learning packages, in addition to mean-squared error.

Note: You won't have to implement the training logger for this algorithm.

In [ ]:
class PolyWeightedLinReg_ClosedForm(Model):
    def __init__(self, degree=1):
        # Note: do not need to modify __init__ method
        self.degree = degree

    def fit(self, Xraw, y, R):
        """ Calculates the closed-form solution parameters for weighted linear regression
            with polynomial features.

            Note: be aware of the shapes of your input when you are doing NumPy operations!

            Input:
            Xraw: np.ndarray of shape (N, M) representing N data points each consisting of M features
            y: np.ndarray of shape (N,1) representing the targets for the each of the N data points
                - data point X[i,:] has target y[i]
                - y is a column vector
            R: np.ndarray of shape (N,) representing the weights for each of the samples

            No output, but self.params is saved with the learned parameters for the model
        """
        ...

    def predict(self, Xraw):
        """ Makes predictions for the new input data Xraw

            Input:
            Xraw: np.ndarray of shape (N, M) representing N data points each consisting of M features

            Output:
            y_hat: np.ndarray of shape (N,1) representing the predicted target values for the each of the datapoints
        """
        ...

    def score(self, Xraw, y):
        ''' Calculates the coefficient of determination by predicting on X and evaluating the metric on y

            Input:
            X: np.ndarray of shape (N, M) representing N data points each consisting of M features
            y: np.ndarray of shape (N, 1) representing the predicted target valeus for each datapoints
        '''
        ...

In [ ]:
Xraw = np.array([1,2,3,4,5,6,7,8]).reshape((-1,1))
y = 8 + 7 * Xraw + 3 * Xraw ** 2 + -3 * Xraw ** 3
R = np.ones((Xraw.shape[0]))

lr = PolyWeightedLinReg_ClosedForm(degree=3)
lr.fit(Xraw, y, R)
score = lr.score(Xraw, y)
score

In [ ]:
Xraw = np.array([[1,1],[2,2],[0,3]])
y = 3 + Xraw @ np.array([1,1]).reshape((-1,1))
R = np.array([1,1,1])

lr = PolyWeightedLinReg_ClosedForm(degree=3)
lr.fit(Xraw, y, R)
predictions = lr.predict(np.array([[1,2],[3,3],[4,5]]))
np.round(predictions, 4)

In [ ]:
grader.check("Q3a")

In [ ]:
def plotModelTemp(model, t):
    '''plot the dataset and the model's predictions on it'''
    data = np.load("data.npz")
    Xraw, y, R = data[f"{t}_X"], data[f"{t}_y"], data[f"{t}_R"]

    model.fit(Xraw, y, np.diag(R))

    Xmin = np.min(Xraw)
    Xmax = np.max(Xraw)
    Xmodel = np.linspace(Xmin, Xmax, 400).reshape((-1,1))
    ymodel = model.predict(Xmodel)

    X_summer, y_summer = data["summer_X"], data["summer_y"]
    X_winter, y_winter = data["winter_X"], data["winter_y"]

    print(f"Score: {model.score(Xraw, y)}")

    plt.figure(figsize=(12,7))
    plt.scatter(X_summer, y_summer, marker="P", c="sandybrown", label="Summer")
    plt.scatter(X_winter, y_winter, marker="P", c="lightskyblue", label="Winter")
    plt.plot(Xmodel, ymodel, linewidth=4, label="model predictions")
    plt.title(f"Temperature vs. Usage Plot with Polynomial Model of Degree {model.degree}")
    plt.xlabel("Temperature")
    plt.ylabel("Usage")
    plt.legend()
    plt.show()

In [ ]:
plotModelTemp(PolyWeightedLinReg_ClosedForm(degree=2), "full")

In [ ]:
plotModelTemp(PolyWeightedLinReg_ClosedForm(degree=5), "full")

In [ ]:
plotModelTemp(PolyWeightedLinReg_ClosedForm(degree=11), "full")

## Question 3b - K-Fold Cross Validation <a class="anchor" name="q3b"></a>

Now that you've created a non-linear model and learned how to fit it to the dataset that you are working with, you feel that you are ready to start training multiple models and choosing the model that you believe will perform the best for the enterprise. However, you would like some way of comparing the generalizability of the models. Recalling different model selection strategies, you decide to perform **k-fold cross validation** to perform model selection, but you will have to implement it from scratch.

There are 3 major parts to performing k-fold cross validation
1. splitting the dataset into k folds
2. computing the training dataset and validation dataset from k folds after choosing a specific fold for validation
3. training, evaluating model across all folds and averaging the metric.

We will have you compute the second two parts of this process for performing k-fold cross validation. Below is the implementation of the first part of the k-fold cross validation process. Do not modify it!

In [ ]:
def createKFolds(Xraw, y, R, k):
    """ Splits/partitions the given dataset into K-fold cross validation datasets usable for model selection.
        Produces k-folds of roughly even sizes of associated data points.

        Input:
        Xraw: np.ndarray of shape (N, M) representing N data points each consisting of M features
        y: np.ndarray of shape (N,1) representing the targets for the each of the N data points
            - data point X[i,:] has target y[i]
            - y is a column vector
        R: np.ndarray of shape (N,) representing the weights for each of the samples
        k: (int) the number of folds to produce in the dataset (k <= Xraw.shape[0])

        Output:
        XrawFolds: list of length k of subset of dataset of Xraw
        yFolds: list of length k of subset of dataset of y
        RFolds: list of length k of subset of dataset of R

    """

    # initialize storage for data
    XrawFolds = []
    yFolds = []
    RFolds = []

    # shuffle the data
    N, M = Xraw.shape

    # construct the folds
    idxs = np.rint(np.linspace(0, N, k+1)).astype(np.int64)
    for i in range(idxs.shape[0] - 1):
        s = idxs[i]
        e = idxs[i+1]
        XrawFolds.append(Xraw[s:e,:])
        yFolds.append(y[s:e,:])
        RFolds.append(R[s:e])
    return XrawFolds, yFolds, RFolds

In [ ]:
# example to see how the k-fold works (feel free to modify this cell block or add any others)
N, M = 100, 3
k = 6
Xraw = np.random.randn(N,M)
y = np.random.randn(N,1)
r = np.random.randn(N)
XrawFolds, yFolds, RFolds = createKFolds(Xraw, y, r, k)

# print the shapes of the folds
print("ith column of shapes represents the shape of the ith fold")
print("X-folds:", end=" "); [print(XrawFolds[i].shape, end=" ") for i in range(len(XrawFolds))]; print()
print("y-folds:", end=" "); [print(yFolds[i].shape, end=" ") for i in range(len(yFolds))]; print()
print("r-folds:", end=" "); [print(RFolds[i].shape, end=" ") for i in range(len(RFolds))]; print()

In [ ]:
def getTrainValFromFold(XrawFolds, yFolds, RFolds, index):
    """ Constructs a training and validation dataset from a k-fold cross validation dataset
        where index is the fold that will be the validation set while the rest are training data.

        Input:
        XrawFolds: list of length k of np.ndarray subsets of the dataset Xraw
        yFolds: list of length k of np.ndarray subsets of target values y
        RFolds: list of length k of np.ndarray subsets of sample weights R
        index: (int) from 0 to k-1 representing the fold that will be the validation dataset

        Output:
        trainXraw: training features composed of vertically stacking the below folds
            - XFolds[0], XFolds[1], ... XFolds[index-1], XFolds[index+1], ..., XFolds[k-1]
        trainy: training targets composed by vertically stacking the below folds
            - yFolds[0], yFolds[1], ... yFolds[index-1], yFolds[index+1], ..., yFolds[k-1]
        trainR: sample weights stored in a 1-dimensional vector composed concatenating the below folds
            - RFolds[0], RFolds[1], ... RFolds[index-1], RFolds[index+1], ..., RFolds[k-1]
        validX: validation features composed of XFolds[index]
        validy: validation targets composed of yFolds[index]
    """
    ...
    return trainXraw, trainy, trainR, validX, validy


In [ ]:
N, M = 20, 3
k = 5
idx = 4
Xraw = np.arange(N*M).reshape((N,M))
y = np.arange(N).reshape((N,1))
r = np.ones(N)
XFolds, yFolds, RFolds = createKFolds(Xraw, y, r, k)
trainX, trainy, trainR, validX, validy = getTrainValFromFold(XFolds, yFolds, RFolds, idx)

assert(trainX.shape[0] == 16)
assert(trainy.shape[0] == 16)
assert(trainR.shape[0] == 16)
assert(validX.shape[0] == 4)
assert(validy.shape[0] == 4)

assert(np.all(validy == yFolds[idx]))
trainX

For the final part of K-Fold Cross Validation, you will need to compute the k-fold cross validation accuracy of a given model that implements `.fit` and `.score`.

In [ ]:
def kFoldCrossValidation(model, XrawFolds, yFolds, RFolds):
    """ Computes the k-fold cross validation score of the given model.
        This validation score is calculated as follows: for each fold, calculate the training and validation datasets
        and average the score across the folds.

        Input:
        model: (Model) implements the `.fit(X, y, R)` method and `.score(X, y)` method that computes
               the coefficient of determination
        XrawFolds: list of length k of np.ndarray subsets of dataset Xraw
        yFolds: list of length k of np.ndarray subsets of target values y
        RFolds: list of length k of np.ndarray subsets of sample weights R

        Output:
        cvScore: the K-Fold Cross Validation score for the input model
    """
    ...

In [ ]:
k = 3
Xraw = np.arange(10).reshape((-1,1))
y = 3 + 5 * Xraw + 3 * np.array([0.2,-0.4,0.4,0.2,0.4,-0.2,0.2,-0.2,0.4,0.2]).reshape((10,1))
r = np.ones(10)
XrawFolds, yFolds, RFolds = createKFolds(Xraw, y, r, k)

model = PolyWeightedLinReg_ClosedForm(degree=3)
score = kFoldCrossValidation(model, XrawFolds, yFolds, RFolds)
np.round(score, 2)

In [ ]:
grader.check("Q3b")

### Q3c - Training and Selecting Models <a class="anchor" name="q3b"></a>

Now that you've completed implementing the k-fold cross validation process, you will train two different types of models, each with 5 different hyperparameters, leading to 10 different models (which we will provide you). You will evaluate the models using k-fold cross validation, and then, you will determine the best model of the 10 from the their cross-validation scores. Note: all of the models will follow the same interface, so they can have the same functions called on them.

The first type of model that you will train is the `PolyWeightedLinReg_ClosedForm` model for `degree=[1,3,5,7,9]`.
The second type of model that you will train is the `KNNRegressor` model for `n_neighbors=[1,3,9,15,25]` (class defined below). We will provide these models for you to compare using cross-validation.

These will be the ten models that you will be training. Note that the `KNNRegressor` just wraps an `sklearn.neighbors.KNeighborsRegressor` model, allowing it to take weighted inputs for fitting but not really using them. This allows the `KNNRegressor` and the `PolyWeightedLinReg_ClosedForm` to have the same interface making them easy to use together in coding.

In [ ]:
def selectModel(models, Xraw, y, R, kFolds=5):
    """ Given a list of models to train (models) and a dataset (Xraw, y, R) and the number of folds (kFolds),
        train and evaluate the models using K-Fold Cross Validation and return their scores and the corresponding best
        model.

        Input:
        models: a list of objects of type Model that are usable for performing K-Fold Cross Validation
        Xraw: np.ndarray of shape (N,M) that is the dataset to train the model with N data points of M features
        y: np.ndarray of shape (N,1) that contains the target values for each sample in the design matrix
        R: np.ndarray of shape (N,) that is an array containing the weights of each sample
        kFolds: (int) number of folds to perform k-fold cross validation for

        Output:
        bestIdx: the index of the model in models with the highest cross-validation score
        cvScores: list of k-fold cross validation scores for input models
            - models[i] has k-fold cross validation score cvScore[i]
    """
    ...


In [ ]:
# DO NOT EDIT this code block
from sklearn.neighbors import KNeighborsRegressor
class KNNRegressor(Model):

    def __init__(self, n_neighbors=5):
        self.knn = KNeighborsRegressor(n_neighbors=n_neighbors)

    def fit(self, Xraw, y, R):
        self.knn.fit(Xraw, y)

    def predict(self, Xraw):
        return self.knn.predict(Xraw).reshape((-1,1))

    def score(self, Xraw, y):
        return self.knn.score(Xraw, y)

# DO NOT EDIT initModels()
def initModels():
        """ Initializes models that is meant for training
        """
        polyModels = [PolyWeightedLinReg_ClosedForm(degree=i) for i in [1,3,5,7,9]]
        knnModels = [KNNRegressor(n_neighbors=i) for i in [1,3,9,15,25]]
        return polyModels + knnModels

In [ ]:
kFolds = 2
Xraw = np.arange(50).reshape((-1,1))
y = 3 + Xraw @ np.array([2]).reshape((-1,1))
R = np.ones(50)

models = initModels()
bestIdx, scores = selectModel(models, Xraw, y, R, kFolds=kFolds)
assert(bestIdx == np.argmax(scores))
np.round(np.array(scores).astype(np.float64).reshape((-1,1)), 2)

In [ ]:
grader.check("Q3c")

### Performing K-Fold Cross Validation with Different Models on Energy Usage Dataset
Run the following commands below to plot the models described on the energy usage dataset.

In [ ]:
def plotModelTemp(model, title="Temperature vs. Usage Plot with Polynomial Model"):
    '''plot the dataset and the model's predictions on it'''
    data = np.load("data.npz")
    Xraw, y, R = data[f"full_X"], data[f"full_y"], data[f"full_R"]

    Xmin = np.min(Xraw)
    Xmax = np.max(Xraw)
    Xmodel = np.linspace(Xmin, Xmax, 400).reshape((-1,1))
    ymodel = model.predict(Xmodel)

    X_summer, y_summer = data["summer_X"], data["summer_y"]
    X_winter, y_winter = data["winter_X"], data["winter_y"]

    plt.figure(figsize=(12,7))
    plt.scatter(X_summer, y_summer, marker="P", c="sandybrown", label="Summer")
    plt.scatter(X_winter, y_winter, marker="P", c="lightskyblue", label="Winter")
    plt.plot(Xmodel, ymodel, linewidth=4, label="model predictions")
    plt.title(title)
    plt.xlabel("Temperature")
    plt.ylabel("Usage")
    plt.legend()
    plt.show()

In [ ]:
# load the dataset and define the number of folds
data = np.load("data.npz")
Xraw, y, R = data[f"full_X"], data[f"full_y"], np.diag(data[f"full_R"])
kFolds = 2

# init the models and train and evaluate using K-Fold Cross Validation
models = initModels()
bestIdx, scores = selectModel(models, Xraw, y, R, kFolds=kFolds)

# plot all models with their dataset
for i, m in enumerate(models):
    if i < 5:
        modelString = f"PolyWLR(degree={m.degree})"
    else:
        n_neighbors = m.knn.get_params()["n_neighbors"]
        modelString = f"KNN(n_neighbors={n_neighbors})"
    title = modelString + f" --- k-fold score: {round(scores[i],2)}"

    # fit model fully to training dataset and then plot
    m.fit(Xraw, y, R)
    plotModelTemp(m, title=title)

## Q3c - Programming Written Plot: plotting the best model <a class="anchor" name="q3c"></a>
Below, we have provided code to plot the best model (according to the k-fold cross validation score). Please provide it in **Q3c** in the writeup of the programming portion of the assignment.

In [ ]:
# of the models above, plot the best model based on k-fold cross validation score
m = models[bestIdx]
if bestIdx < 5:
    modelString = f"PolyWLR(degree={m.degree})"
else:
    n_neighbors = m.knn.get_params()["n_neighbors"]
    modelString = f"KNN(n_neighbors={n_neighbors})"
title = modelString + f" --- k-fold score: {round(scores[i],2)}"

# fit model fully to training dataset and then plot
m.fit(Xraw, y, R)
plotModelTemp(m, title=title)

### Submit your code to Gradescope early and often

There is no limit on the number of submissions to Gradescope, so as you complete parts of the assignment it is a really good idea to save your notebook and upload it to Gradescope.

Not all of the tests are included in the local autograder. Some of the tests are "hidden" and only run in the server autograder on Gradescope.

Before continuing with the rest of the assignment, go ahead and save your notebook (or click File->Download->Download .ipynb) and then upload your hw3.ipynb file to Gradescope under assignment HW3 (programming).